# Strategy sandbox — <ID> <name>

Template for exploring a signal before it becomes `strategies/<ID>/strategy.py`.

Rules:
- This notebook is a SCRATCHPAD. Nothing here counts as a deliverable.
- Once the signal is settled, port it to `strategy.py` — that is what ships.
- Do not put costs in this notebook. Explore gross here; costs happen in the real run.

In [ ]:
import sys, pandas as pd, numpy as np
sys.path.insert(0, '.')
from fetcher import load_parquet
from metrics import tear_sheet, format_tear_sheet
from backtester import backtest, buy_and_hold, CostModel

px = load_parquet('data/clean/aapl.parquet')['adj_close']
print(len(px), px.index[0].date(), px.index[-1].date())
px.tail()

## 1. Look at the data before you model it

Plot it. Count NaNs. Check for duplicate dates and for gaps. Most 'mysterious'
backtest results are data problems wearing a costume.

In [ ]:
assert px.index.is_unique, 'duplicate dates'
assert px.index.is_monotonic_increasing, 'not sorted'
print('NaNs:', px.isna().sum())
print('gaps > 5 days:', (px.index.to_series().diff().dt.days > 5).sum())
px.plot(figsize=(11, 3), title='price', logy=True);

## 2. Build the signal

Write it so it uses only data available AT OR BEFORE bar t. Then prove it with
the cell below.

In [ ]:
def signal(prices, fast=10, slow=50):
    out = (prices.rolling(fast).mean() > prices.rolling(slow).mean()).astype(float)
    out[prices.rolling(slow).mean().isna()] = 0.0
    return out

sig = signal(px)
print('fraction long:', (sig > 0).mean().round(3))
print('flips:', int(sig.diff().abs().gt(0).sum()))

## 3. Look-ahead check (do not skip)

Rewrite the LAST 100 bars of prices (the future). If any signal value in the
FIRST part of the series changes, your signal peeks.

In [ ]:
px2 = px.copy(); px2.iloc[-100:] *= 1.5
sig2 = signal(px2)
n = len(px) - 100
diff = (sig.iloc[:n] != sig2.iloc[:n]).sum()
print('signal values changed by rewriting the future:', int(diff))
assert diff == 0, 'LOOK-AHEAD: your signal uses future data'

## 4. Gross backtest, then net

Gross is a diagnostic, not a result. Net is the result.

In [ ]:
gross = backtest(px, sig, cost_model=CostModel(0, 0, 0))
net   = backtest(px, sig, cost_model=CostModel(1, 5, 50))
bh    = buy_and_hold(px, cost_model=CostModel(1, 5, 50))
print(gross.summary('GROSS'))
print(net.summary('NET'))
print(bh.summary('BUY & HOLD'))

In [ ]:
res = net
pd.DataFrame({'equity': res.equity, 'position': res.position}).plot(
    subplots=True, figsize=(11, 5), title=['equity', 'position']);

## 5. Now port it

Move `signal()` into `strategies/<ID>/strategy.py` with a `--fast/--slow` CLI,
run it from the shell, and write the tear sheet to `tearsheet.txt`.
The notebook is not evidence. The script plus its output is.